# Extract

## WTI Oil Prices
* Lag features
#### Economic growth
* US dollar index 
* S&P 500 index 
* US Federal Reserve rate
* Volatility index (VIX)
* Consumer price index (CPI).

source: https://www.investopedia.com/articles/investing/072515/top-factors-reports-affect-price-oil.asp


EIA
* wti_prices -> daily
* oil_production -> monthly
* input_utilization -> weekly
* gasoline_price -> weekly
* imports_and_exports -> weekly
* weekly_stocks -> weekly

FRED
* us_dollar_index -> daily
* volatility_index -> daily
* cpi_energy -> monthly
* s&p500 -> daily

## Energy Information Administration (EIA) Api
| Features | Frequence | Data Until |
|---|---|---|
| WTI | Daily | up-to-date |
| Oil Production | Monthly | July 2025|
|Weekly Input Utilization | Weekly | up-to-date |
| Gasoline Price | Weekly | up-to-date |
|Imports & Exports | Weekly | up-to-date |
|Crude Oil Supplied | Monthly | July 2025 |

In [4]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

In [2]:
%pwd

'/workspaces/oil-optimization'

In [4]:
import pandas as pd
import requests
from dotenv import dotenv_values

secrets = dotenv_values('.env')

api_calls  = {
    'wti_prices':{
        'url':'https://api.eia.gov/v2/petroleum/pri/spt/data',
        'payload':{
                    'api_key':secrets['EIA_API_KEY'],
                    'facets[product][]':'EPCWTI',
                    'start':'2015-01-01',
                    'sort[0][column]':'period',
                    'sort[0][direction]':'asc',
                    'frequency':'daily',
                    'data[0]':'value'
                }
    },
    'oil_production':{
        'url':'https://api.eia.gov/v2/petroleum/crd/crpdn/data',
        'payload':{
                    'api_key':secrets['EIA_API_KEY'],
                    'facets[product][]':'EPC0',
                    'start':'2015-01-01',
                    'sort[0][column]':'period',
                    'sort[0][direction]':'asc',
                    'data[0]':'value'
                },
        'date_intervals':[
                    ('2015-01','2020-02'),
                    ('2020-02','2025-03'),
                    ('2025-03','')
                    ]
    },
    'input_utilization':{
        'url':'https://api.eia.gov/v2/petroleum/pnp/wiup/data',
        'payload':{
                    'api_key':secrets['EIA_API_KEY'],
                    'facets[product][]':'EPC0',
                    'start':'2015-01-01',
                    'sort[0][column]':'period',
                    'sort[0][direction]':'asc',
                    'data[0]':'value'
                }
    },
    'gasoline_price':{
        'url':'https://api.eia.gov/v2/petroleum/pri/gnd/data',
        'payload':{
                    'api_key':secrets['EIA_API_KEY'],
                    'facets[product][]':'EPM0',
                    'start':'2015-01-01',
                    'sort[0][column]':'period',
                    'sort[0][direction]':'asc',
                    'data[0]':'value',
                },
        'date_intervals':[
                    ('2015-01-01','2017-12-31'),
                    ('2018-01-01','2021-03-31'),
                    ('2021-04-01','2024-06-30'),
                    ('2024-07-01',None)
                    ]
    },
    'imports_and_exports':{
        'url':'https://api.eia.gov/v2/petroleum/move/wkly/data',
        'payload':{
                    'api_key':secrets['EIA_API_KEY'],
                    'facets[product][]':'EPC0',
                    'start':'2015-01-01',
                    'sort[0][column]':'period',
                    'sort[0][direction]':'asc',
                    'data[0]':'value'
                },
        'date_intervals':[
                    ('2015-01-01','2023-08-31'),
                    ('2023-09-01','')
                    ]
    },
    'oil_supplied':{
        'url':'https://api.eia.gov/v2/petroleum/cons/psup/data',
        'payload':{
                    'api_key':secrets['EIA_API_KEY'],
                    'facets[product][]':'EPC0',
                    'start':'2015-01-01',
                    'sort[0][column]':'period',
                    'sort[0][direction]':'asc',
                    'data[0]':'value'
                }
    }}

In [32]:
from src.oil_optimization.utils.io_helpers import read_yaml

config = read_yaml('config/config.yml')
api_config = read_yaml('config/api_config.yml')
eia_api_calls = api_config['eia_api']

data_dir = config['data_ingestion']['data_dir']

In [ ]:
from abc import ABC, abstractmethod
from typing import Any
import time
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from requests.packages.urllib3.util.retry import Retry
from dotenv import dotenv_values
from src.oil_optimization.utils.io_helpers import read_yaml, save_csv

SECRETS = dotenv_values('.env')
EIA_API_KEY = SECRETS['EIA_API_KEY']
FRED_API_KEY = SECRETS['FRED_API_KEY']


In [79]:
import logging
import sys
from abc import ABC, abstractmethod
from typing import Any
import time
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from requests.packages.urllib3.util.retry import Retry
from dotenv import dotenv_values
from src.oil_optimization.utils.io_helpers import read_yaml, save_csv

SECRETS = dotenv_values('.env')
EIA_API_KEY = SECRETS['EIA_API_KEY']
FRED_API_KEY = SECRETS['FRED_API_KEY']

logging.basicConfig(
                    level=logging.INFO,
                    format="%(asctime)s | %(name)s | %(levelname)s | %(lineno)d | %(message)s",
                    handlers=[
                    logging.FileHandler("myapp.log"),
                    logging.StreamHandler(sys.stdout),
                    ],
                    force=True
                    )
logger = logging.getLogger(__name__)


class BaseExtractor(ABC):
    def __init__(self):
        super().__init__()
        self.config = read_yaml("config/config.yml")
        self.data_dir = self.config["data_ingestion"]["data_dir"]
        self.api_config = read_yaml("config/api_config.yml")
        self.session = requests.Session()

        adapter = HTTPAdapter(max_retries=3)
        self.session.mount("https://", adapter)

        logger.info("Http Session Initiated")

    def get(self, url: str, method: str = "GET", payload: dict[str,Any] = {}):
        try:
            if not payload:
                logger.warning("No query parameters are being sent to %s due to empty payload", url)
            
            
            r = self.session.request(method=method,
                                    url=url,
                                    timeout=30,
                                    params=payload)
            logger.info("Status code: %s", r.status_code)
            r.raise_for_status()
            time.sleep(3)
            return r.json()
        except requests.exceptions.HTTPError as e:
            logger.exception(e)
    
    @abstractmethod
    def extract_data(self, response_data):
        pass

    def save_to_csv(self, df: pd.DataFrame, filename: str):
        path = f'{self.data_dir}/raw/{filename}.csv'
        df.to_csv(path, index=False)
        logger.info("%s.csv successfully created!", filename)

class EIAExtractor(BaseExtractor):
    def __init__(self) -> None:
        super().__init__()
        self.eia_api_config = self.api_config['eia_api']

    def extract_data(self, response_data):
        return response_data['response']['data']

    def create_file(self, label:str, params:dict[str,Any]):
        data_list = []
        params['payload']['api_key'] = EIA_API_KEY
        if "date_intervals" in params.keys():
            for i, date in enumerate(params['date_intervals']):
                payload = params['payload'].copy()
                payload['start'] = date[0]

                if date[1]:
                    payload['end'] = date[1]
                if i == 0:
                    logger.info("Limited data retrieval from %s due to API limits, starting pagination...", params["url"])
                logger.info("Sending HTTP request #%s", i+1)
                data = self.get(params["url"], payload=payload)
                json_data = self.extract_data(data)

                data_list += json_data
                
            df = pd.DataFrame(data_list)
            self.save_to_csv(df, label)

        else:
            data = self.get(params['url'], payload=params['payload'])
            json_data = self.extract_data(data)
            df = pd.DataFrame(json_data)
            self.save_to_csv(df, label)

In [80]:
extractor = EIAExtractor()

label = "wti_prices"
payload = extractor.api_config["eia_api"][label]
r_json = extractor.create_file(label=label, params=payload)
extractor.session.close()

2026-04-05 03:39:47,848 | __main__ | INFO | 40 | Http Session Initiated
2026-04-05 03:39:47,850 | __main__ | INFO | 87 | Limited data retrieval from https://api.eia.gov/v2/petroleum/pri/spt/data due to API limits, starting pagination...
2026-04-05 03:39:47,851 | __main__ | INFO | 88 | Sending HTTP request #1


2026-04-05 03:39:48,827 | __main__ | INFO | 52 | Status code: 200
2026-04-05 03:39:51,842 | __main__ | INFO | 88 | Sending HTTP request #2
2026-04-05 03:39:52,192 | __main__ | INFO | 52 | Status code: 200
2026-04-05 03:39:55,232 | __main__ | INFO | 66 | wti_prices.csv successfully created!


In [57]:
if None:
    print(True)

## Stock Market Indexes
* S&P 500 Energy

In [ ]:
import yfinance

yfinance.download('^GSPE')

## Federeal Reserve Bank of St. Louis (FRED)
* US Dollar Index (DXY)
* Volatility Index (VIX)
* Consumer Price Index (CPI)

In [6]:
from oil_optimization.utils.io_helpers import read_yaml
from dotenv import dotenv_values

secrets = dotenv_values('.env')

fred_api = read_yaml('config/api_config.yml')['fred_api']

In [72]:
url = f'https://api.stlouisfed.org/fred/series/observations?series_id={'DTWEXBGS'}&api_key={secrets['FRED_API_KEY']}&file_type=json&observation_start=2015-01-01'

r = requests.get(url=url)


In [73]:
list_data = []
for item in r.json()['observations']:
    list_data.append({'date':item['date'],
           'value':item['value']})

In [74]:
pd.DataFrame(list_data)

,date,value
0,2015-01-01,.
1,2015-01-02,102.9027
2,2015-01-05,103.4976
3,2015-01-06,103.2938
4,2015-01-07,103.6316
...,...,...
2812,2025-10-13,.
2813,2025-10-14,121.5815
2814,2025-10-15,121.2669
2815,2025-10-16,121.0834


In [1]:
import os
os.chdir('..')
from dotenv import dotenv_values
import pandas as pd

SECRETS = dotenv_values('.env')
EIA_API_KEY = SECRETS['EIA_API_KEY']
FRED_API_KEY = SECRETS['FRED_API_KEY']

In [4]:
from src.oil_optimization.data_pipeline.extractor import EIAExtractor, FREDExtractor
from dotenv import dotenv_values
import pandas as pd

SECRETS = dotenv_values('.env')
EIA_API_KEY = SECRETS['EIA_API_KEY']
FRED_API_KEY = SECRETS['FRED_API_KEY']

eia_extractor = EIAExtractor()
for key, params_dict in eia_extractor.eia_api_config.items():
    eia_extractor.create_file(label=key, params=params_dict)

fred_extractor = FREDExtractor()
for key, params_dict in fred_extractor.fred_api_config.items():
    params_dict['api_key'] = FRED_API_KEY
    fred_json = fred_extractor.make_request(url=fred_extractor.url, payload=params_dict)
    fred_data = fred_extractor.extract_data(fred_json)
    dataframe = pd.DataFrame(fred_data). \
        rename({'date':'period'},axis=1) # Change of date name from 'date' to 'period'
    fred_extractor.save_to_csv(dataframe, key)

Label 1: wti_prices
200
Label 2: wti_prices
200
Datafile wti_prices saved
Label 1: oil_production
200
Label 2: oil_production
200
Label 3: oil_production
200
Datafile oil_production saved
200
Datafile input_utilization saved
Label 1: gasoline_price
200
Label 2: gasoline_price
200
Label 3: gasoline_price
200
Label 4: gasoline_price
200
Datafile gasoline_price saved
Label 1: imports_and_exports
200
Label 2: imports_and_exports
200
Datafile imports_and_exports saved
200
Datafile weekly_stocks saved
200
Datafile us_dollar_index saved
200
Datafile volatility_index saved
200
Datafile cpi_energy saved
200
Datafile sp500 saved


# Logging

In [35]:
import logging
import numpy as np
import sys

logging.basicConfig(filename="trial.log", level=logging.INFO)

logger = logging.getLogger(__name__)
logging.basicConfig(
                    level=logging.INFO,
                    format="%(asctime)s | %(name)s | %(levelname)s | %(lineno)d | %(message)s",
                    handlers=[
                    logging.FileHandler("myapp.log"),
                    logging.StreamHandler(sys.stdout),
                    ],
                    force=True
                    )
logger.info("Trial started")
print(np.array([2,3,4]) * 4)
logger.info("Trial finished")

2026-04-04 21:11:24,179 | __main__ | INFO | 17 | Trial started
[ 8 12 16]
2026-04-04 21:11:24,182 | __main__ | INFO | 19 | Trial finished


In [36]:
logging.root.handlers

[<FileHandler /workspaces/oil-optimization/notebooks/myapp.log (NOTSET)>,
 <StreamHandler stdout (NOTSET)>]